# ISIC 2019 setup

This notebook creates the concrete Bronze file layout and tables for ISIC 2019.

Landing stays shared and only holds the downloaded source archive.

In [ ]:
%run ../_setup_env


In [ ]:
from pprint import pprint

from isic_pipeline.setup import build_layout, load_dataset_config, load_yaml_config

DATASET_CONFIG = load_dataset_config(CONFIG_ROOT / "datasets" / "isic_2019.yaml")
STORAGE_CONFIG = load_yaml_config(CONFIG_ROOT / "storage.yaml")
DATASET_CONFIG["landing_root"] = STORAGE_CONFIG["landing_root"]
DATASET_CONFIG["storage_root"] = STORAGE_CONFIG["storage_root"]
layout = build_layout(DATASET_CONFIG)
pprint(layout)


In [ ]:
spark.sql(
    """
    CREATE TABLE IF NOT EXISTS bronze.isic_2019_source_metadata (
      image_id STRING,
      source_uri STRING,
      patient_id STRING,
      lesion_id STRING,
      label_raw STRING,
      label_source STRING,
      label_normalized STRING,
      acquisition_date STRING,
      source_checksum STRING,
      ingestion_run_id STRING,
      ingested_at TIMESTAMP
    )
    USING DELTA
    """
)

display(spark.sql("SHOW TABLES IN bronze"))


In [ ]:
landing_paths = layout["landing_paths"]
bronze_paths = layout["bronze_paths"]

dbutils.fs.mkdirs(landing_paths["dataset_archive"])

for path in bronze_paths.values():
    dbutils.fs.mkdirs(path)

display(dbutils.fs.ls(landing_paths["archives"]))
display(dbutils.fs.ls(bronze_paths["root"]))


## Next step

Use this notebook as the first concrete ISIC 2019 Bronze setup.
Keep the downloaded archive in landing, then materialize extracted images and metadata into Bronze paths.